In [ ]:
import numpy as np
from qutip import Bloch
import numba
from numba import njit, prange
import pickle
import os
import shutil
import imageio
from IPython.display import Image, display
from multiprocessing import Pool, cpu_count
from functools import partial
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

In [ ]:

sz = np.array(([[1,0], [0,-1]]), dtype=complex); sx = np.array(([[0,1],[1,0]]), dtype=complex); sy = np.array(([[0,-1j],[1j,0]]), dtype=complex) 


In [ ]:
# ======================================
# Example to choose the Axis orientation
# ======================================

# ! Comment matplotlib.use('Agg') and restar kernell to run !

b = Bloch()
b.view = [-60, 30]  # Azimuthal angle, High form xy plane

b.show()

In [ ]:
MODE = "QJ"  # or "SD"

# -----------------------------
# Input Directory and file name
# -----------------------------
input_path = "../../Results/Data/Complete_rho"
if MODE == "QJ":
    input_file = os.path.join(input_path, f"result_mode_QJ_dt0p010000_Ntraj20000.npz")
elif MODE == "SD":
    input_file = os.path.join(input_path, f"result_mode_SD_dt0p010000_Ntraj1000.npz")
 
# ------------
# Load Results
# ------------
data = np.load(input_file)

# -----------------
# Select parameters
# -----------------

dt_value = 0.01
N_traj_value = 1000
trajectory_idx = 0  # Which trajectory sample to plot
sample_step = 10    # Sample every N steps

# ------------
# Extract data
# ------------

# Extract Lindblad populations
rho_list_lindblad = data['rho_list_lindblad']
lindblad_00 = np.real(rho_list_lindblad[:, 0, 0])
lindblad_11 = np.real(rho_list_lindblad[:, 1, 1])
lindblad_01 = rho_list_lindblad[:, 0, 1]

avg_pop_00 = np.mean(data['pop_00'][::sample_step], axis=1)
avg_pop_11 = np.mean(data['pop_11'][::sample_step], axis=1)
avg_coh_01 = np.mean(data['coh_01'][::sample_step], axis=1)

# Extract 10 samples over all trajectories, to plot them together on the same Bloch sphere
sample_traj_pop_00 = data['pop_00'][::sample_step, :10]  # First 10 trajectories
sample_traj_pop_11 = data['pop_11'][::sample_step, :10]  # First 10 trajectories
sample_traj_coh_01 = data['coh_01'][::sample_step, :10]  # First 10 trajectories

# Extract ONLY 1 sample trajectory using trajectory_idx
# sample_traj_pop_00 = data['pop_00'][::sample_step, trajectory_idx]  
# sample_traj_pop_11 = data['pop_11'][::sample_step, trajectory_idx]  
# sample_traj_coh_01 = data['coh_01'][::sample_step, trajectory_idx]

# ------------------------------
# Output Directory and file name
# ------------------------------
output_path = "../../Results/Bloch_Sphere"
os.makedirs(output_path, exist_ok=True)

# Define the final path for the GIF file
gif_path = os.path.join(output_path, f"bloch_animation_{MODE}_dt{dt_value}_N{N_traj_value}.gif")

# --------------------
# Temporary Directory
# --------------------
temp_dir = os.path.join(output_path, 'temp_bloch_frames')

if os.path.exists(temp_dir):
    shutil.rmtree(temp_dir)
os.makedirs(temp_dir)

In [ ]:
# ==========================================
# Compute Bloch Sphere Components (x, y, z)
# ==========================================

# 1. Lindblad (Analytical Baseline)
# ---------------------------------
lindblad_x = 2 * np.real(lindblad_01)
lindblad_y = -2 * np.imag(lindblad_01)
lindblad_z = lindblad_00 - lindblad_11

# 2. Stochastic Average
# ---------------------
# Note: Using your variable names. If MODE="SD", these actually hold SD data.
avg_x = 2 * np.real(avg_coh_01_QJ)
avg_y = -2 * np.imag(avg_coh_01_QJ)
avg_z = avg_pop_00_QJ - avg_pop_11_QJ

# 3. Sample Trajectories
# ----------------------
# These arrays will have shape (N_times, N_samples)
v_x = 2 * np.real(sample_traj_coh_01_QJ)
v_y = -2 * np.imag(sample_traj_coh_01_QJ)
v_z = sample_traj_pop_00_QJ - sample_traj_pop_11_QJ

In [ ]:
# ========================================
# Generate Bloch Sphere Animation
# ========================================

def generate_single_frame(i, v_x, v_y, v_z, avg_x, avg_y, avg_z, temp_dir, n_steps):
    """
    Generate a single frame of the Bloch sphere animation.
    """
    try:        
        sphere = Bloch()
        sphere.view = [-60, 30]
            
        # Add points with fading effect
        for j in range(i + 1):    # j=0 first point, till last point in time i+1
            sphere.add_points([v_x[j], v_y[j], v_z[j]], alpha=np.exp((j - i) / 15))
            
        # Add average vector
        sphere.add_vectors([avg_x[i], avg_y[i], avg_z[i]])
            
        # Configuration
        sphere.point_color = 'b'
        sphere.point_marker = 'o'
        sphere.point_size = [25]
            
        # Generate and save
        sphere.make_sphere()
        frame_name = f'bloch_{str(i).zfill(len(str(n_steps)))}.png'
        frame_path = os.path.join(temp_dir, frame_name)
        sphere.fig.savefig(frame_path, dpi=80, bbox_inches='tight')
        plt.close(sphere.fig)
            
        return frame_path

    except Exception as e:
        print(f"Error on frame {i}: {e}")
        return None

n_steps = len(v_x)  # number of frames to generate
n_processes = 12  # number of CPU cores to use

print(f"\nTotal frames to generate: {n_steps}")
print(f"Using {n_processes} parallel processes")
print(f"Generating frames...")

# Parallel frame generation
generate_frame_partial = partial(
    generate_single_frame, 
    v_x=v_x, v_y=v_y, v_z=v_z,
    avg_x=avg_x, avg_y=avg_y, avg_z=avg_z, 
    temp_dir=temp_dir, n_steps=n_steps
)

with Pool(processes=n_processes) as pool:
    frame_paths = pool.map(generate_frame_partial, range(n_steps))

# Create GIF 
print("Creating GIF...")
filenames = sorted([os.path.join(temp_dir, f) for f in os.listdir(temp_dir) if f.endswith('.png')])

with imageio.get_writer(gif_path, mode='I', duration=0.1) as writer:   # duration of each frame in seconds
    for filename in filenames:
        image = imageio.imread(filename)
        writer.append_data(image)

# Clean up temporary files
shutil.rmtree(temp_dir)

# Display result
file_size_mb = os.path.getsize(gif_path) / (1024 * 1024)
print(f"\nGIF saved to: {gif_path}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Number of frames: {n_steps}")
print(f"Duration: {n_steps * 0.1:.2f} seconds")
print(f"Temporary files cleaned up")

In [ ]:
display(Image(filename=gif_path))